# Linear Regression — Mock Final Exam Practice 🎓
## Mr Beast Videos Dataset

### Style & Rules (mimicking the professor's quiz style)
- Use Python 3.13.5
- Use **alpha = 0.05** unless stated otherwise
- Use **`random_state = 1031`** wherever relevant (different seed!)
- Use the **`train`** sample for analysis unless stated otherwise
- Do NOT round answers
- No commas in numbers (`100000`, not `100,000`)
- No units (`34.56`, not `$34.56`)
- Lead with 0 (`0.34`, not `.34`)
- Drop trailing zeros (`0.3`, not `0.30`)

### Differences from the original quiz
- **Target** changed: predict `views` instead of `likes`
- **Split ratio** changed: `80/20` instead of `90/10`
- **Seed** changed: `1031` instead of `617`
- Includes a few harder questions on coefficient interpretation

**👉 Try each question yourself FIRST. Run the code only after writing your own answer!**

---
## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

data = pd.read_csv('mr_beast.csv')

# Apply ordered categorical (Mon as reference)
data['publish_day'] = pd.Categorical(data['publish_day'],
                                      categories=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                                      ordered=True)

# Different setup from original quiz: 80/20 split, seed=1031
train = data.sample(frac=0.8, random_state=1031)
test = data.drop(labels=train.index)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

---
## ❓ Question 1

> Read in the data and apply `pd.Categorical()` to convert publish_day to an ordered categorical variable with Mon as the first day of the week.
>
> Next, split the data into a train and test sample such that **80%** of the data is in the train sample. Use simple random sampling with `data.sample()`. For reproducibility set random_state to **1031**.
>
> **What is the mean number of `views` in the train sample?**

### ✅ Answer: `22832167.539130434`

In [ ]:
train.views.mean()
# → 22832167.539130434

---
## ❓ Question 2

> Construct a scatter plot to examine the relationship between `views` and `comments`. Place `comments` on the x-axis and `views` on the y-axis.
>
> **What is the direction of the points?**

### ✅ Answer: **Upward / Positive** — points trend from bottom-left to top-right.

Correlation = **0.7512** (strong positive)

In [ ]:
sns.scatterplot(data=train, x='comments', y='views', color='steelblue', s=15)
plt.show()

print('Correlation:', train[['comments','views']].corr().iloc[0,1])

---
## ❓ Question 3

> **Which of the following variables have a statistically significant pearson correlation with `views`?**

### ✅ Answer: **`likes`, `comments`, `number_of_tags`, `time_since`** (all have p < 0.05)

| Variable | Correlation | p-value | Significant? |
|---|---:|---:|:---:|
| likes | 0.9344 | 4.05e-259 | ✅ YES |
| comments | 0.7512 | 1.95e-105 | ✅ YES |
| duration_seconds | 0.0219 | 0.600 | ❌ NO |
| number_of_tags | -0.3931 | 1.10e-22 | ✅ YES |
| length_description | 0.0162 | 0.699 | ❌ NO |
| length_title | 0.0801 | 0.055 | ❌ NO (just barely!) |
| time_since | -0.8009 | 1.01e-129 | ✅ YES |

In [ ]:
numeric_vars = ['likes','comments','duration_seconds','number_of_tags',
                'length_description','length_title','time_since']
for var in numeric_vars:
    r, p = pearsonr(train[var], train['views'])
    sig = 'YES' if p < 0.05 else 'NO'
    print(f'{var:<25} r={r:>8.4f}  p={p:.4e}  sig={sig}')

---
## ❓ Question 4

> **Which of the following variables has the WEAKEST (smallest |r|) pearson correlation with `views`?**

### ✅ Answer: **`length_description`** (|r| = 0.0162)

⚠️ Notice the trick: question asks WEAKEST, not STRONGEST. Read carefully!

In [ ]:
results = []
for var in numeric_vars:
    r, p = pearsonr(train[var], train['views'])
    results.append((var, r, abs(r)))
results.sort(key=lambda x: x[2])  # ascending — weakest first
for v, r, ar in results:
    print(f'{v:<25} r={r:>8.4f}  |r|={ar:.4f}')

---
## ❓ Question 5

> One might expect that longer videos retain viewers' attention and could lead to more views. Construct a regression model to predict `views` using `duration_seconds`. Call this `model_a`.
>
> **Examine the p-value for `duration_seconds`. Is the coefficient statistically significant?**

### ✅ Answer: **NOT statistically significant** (p ≈ 0.600 > 0.05)

We FAIL to reject H₀: β₁ = 0. There is no statistical evidence that `duration_seconds` linearly affects `views`.

In [ ]:
model_a = ols('views ~ duration_seconds', data=train).fit()
print(model_a.summary())

---
## ❓ Question 6

> **Based on the coefficient of `duration_seconds` in `model_a`, which of the following is true?**

### ✅ Answer: **The coefficient is approximately 135.40 (positive); each additional second is associated with a 135.40 increase in predicted views.**

⚠️ But remember from Q5 — this effect is **not statistically significant**. Some statistically-oriented analysts would say there is *no reliable evidence* of an effect, even though we can compute a numeric prediction.

In [ ]:
model_a.params

---
## ❓ Question 7

> **Based on `model_a`, what is the predicted views for a video with `duration_seconds = 600`?**

### ✅ Answer: **`22767003.534707773`**

In [ ]:
model_a.predict(pd.DataFrame({'duration_seconds':[600]}))

---
## ❓ Question 8

> The first video in the train data has `2741710` views and `duration_seconds = 126`. **Based on `model_a`, what is the predicted number of views for the first video?**

### ✅ Answer: **`22702824.52676366`**

💡 Notice: actual = 2,741,710 but predicted = 22,702,824. **Huge residual!** This is because `duration_seconds` is a poor predictor (R² ≈ 0.0005).

In [ ]:
# Predict for first video using statsmodels
model_a.predict(train.iloc[[0]])

# Manual
b0 = model_a.params['Intercept']
b1 = model_a.params['duration_seconds']
print('Manual:', b0 + b1 * train.iloc[0]['duration_seconds'])

---
## ❓ Question 9

> Construct a model to examine the effect of `length_title` on `views`. Call this `model_b`. **Based on `model_b`, what is the effect of adding 5 more characters to a video title on views?**

### ✅ Answer: **An increase of approximately `1156276.18` views** (i.e., 5 × 231,255.24)

⚠️ Tricky question — asks for effect of **5** units, not 1!  
Coefficient = `231255.24` per character → for 5 characters: 5 × 231255.24 = **`1156276.181...`**

In [ ]:
model_b = ols('views ~ length_title', data=train).fit()
print(model_b.params)
print('p-value:', model_b.pvalues['length_title'])
print('\nEffect of 5 more characters:', 5 * model_b.params['length_title'])

---
## ❓ Question 10

> Run a linear regression to examine the influence of `publish_day` on `views`. Call this `model_c`. **What is the R² of this model?**

### ✅ Answer: **`0.12899783299327...`**

In [ ]:
model_c = ols('views ~ publish_day', data=train).fit()
print('R²:', model_c.rsquared)
print(model_c.summary())

---
## ❓ Question 11

> **Based on `model_c`, on average how many more views do videos released on Saturday get than videos released on Monday?**

### ✅ Answer: **`39525807.882388055`**

In [ ]:
model_c.params['publish_day[T.Sat]']
# → 39525807.882388055

---
## ❓ Question 12

> Based on `model_c`, **which of the following days are statistically significantly different from Monday at α=0.05?** Select one or more.

### ✅ Answer: **Thu, Fri, Sat**

| Day vs Mon | Coefficient | p-value | Sig? |
|---|---:|---:|:---:|
| Tue | +13,239,071 | 0.055 | ❌ NO (just barely!) |
| Wed | +5,635,850 | 0.425 | ❌ NO |
| Thu | +21,580,328 | 0.001 | ✅ YES |
| Fri | +21,074,212 | 0.001 | ✅ YES |
| Sat | +39,525,808 | <0.001 | ✅ YES |
| Sun | +2,235,241 | 0.730 | ❌ NO |

⚠️ Note: Tue p = 0.0549 → just above 0.05 threshold → NOT significant. The professor often plants questions like this to test whether you understand the threshold strictly!

In [ ]:
print(model_c.pvalues)
print()
print('Significant days (p<0.05):')
for name, p in model_c.pvalues.items():
    if name != 'Intercept' and p < 0.05:
        print(f'  {name}: p={p:.4f}')

---
## ❓ Question 13

> In this section, you will use the scikit-learn framework. Unless stated otherwise, use the train sample for analysis.
>
> Separate the outcome variable from the features, assigning `views` to a Series, `y`, and assigning the following features to a DataFrame, `X`: `likes, comments, duration_seconds, number_of_tags, length_description, length_title, time_since, publish_day`.
>
> Next, create a binned outcome variable as follows:
> ```python
> y_binned = pd.cut(data.views, bins=[0, 100000, 1000000, 10000000, 100000000, 1000000000])
> ```
>
> Use `y_binned` to do stratified random sampling, placing **80%** in train and **20%** in test. Set `random_state=1031`.
>
> **What is the 25th percentile (Q1) of `comments` in the train sample?**

### ✅ Answer: **`279.5`**

In [ ]:
y = data.views
X = data[['likes','comments','duration_seconds','number_of_tags',
          'length_description','length_title','time_since','publish_day']]

y_binned = pd.cut(data.views, bins=[0, 100000, 1000000, 10000000, 100000000, 1000000000])

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X, y, train_size=0.8, random_state=1031, stratify=y_binned)

print('Train shape:', X_tr_raw.shape)
print('Q1 (25th percentile) of comments:', X_tr_raw['comments'].quantile(0.25))
# → 279.5

---
## ❓ Question 14

> Dummy code `publish_day` using `pandas.get_dummies()` with `drop_first=True`. Do this for both train and test samples.
>
> **What is the mean value for the dummy variable representing Saturday?**

### ✅ Answer: **`0.23652173913043478`**

💡 ~24% of train videos were published on Saturday — Mr Beast's preferred day!

In [ ]:
X_tr = pd.get_dummies(X_tr_raw, columns=['publish_day'], drop_first=True).astype(float)
X_te = pd.get_dummies(X_te_raw, columns=['publish_day'], drop_first=True).astype(float)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

print('Mean publish_day_Sat:', X_tr['publish_day_Sat'].mean())
# → 0.23652173913043478

---
## ❓ Question 15

> Fit a regression model to predict `views` using all features in the matrix above. Call this `model_d`. **What is the predicted value for `views` for the 5th observation in the train sample?**

### ✅ Answer: **`4999702.709397368`**

💡 Use `iloc[4]` (0-indexed) to get the 5th observation.

In [ ]:
model_d = LinearRegression()
model_d.fit(X_tr, y_tr)
y_pred_tr = model_d.predict(X_tr)
y_pred_te = model_d.predict(X_te)

# 5th observation = iloc[4] (zero-indexed)
print('Predicted views for 5th obs:', y_pred_tr[4])
# → 4999702.709397368

---
## ❓ Question 16

> **What is the root mean squared error (RMSE) for `model_d` in the train sample?**

### ✅ Answer: **`18123259.226605773`**

In [ ]:
rmse_tr = root_mean_squared_error(y_tr, y_pred_tr)
print('Train RMSE:', rmse_tr)
# → 18123259.226605773

---
## ❓ Question 17

> **What is the root mean squared error (RMSE) for `model_d` in the test sample?**

### ✅ Answer: **`17554806.20928597`**

In [ ]:
rmse_te = root_mean_squared_error(y_te, y_pred_te)
print('Test RMSE:', rmse_te)
# → 17554806.20928597

---
## ❓ Question 18 (BONUS — interpretation)

> Examine the coefficients of `model_d`. **Which feature has the largest absolute coefficient?** 
>
> *Bonus question*: Does this mean it is the most important predictor of views?

### ✅ Answer: **`publish_day_Sat`** has the largest absolute coefficient (~9,794,992)

**However, the answer to whether it is the *most important* predictor is NO — and this is the trick!** Reasons:
1. Coefficient size depends on the **scale** of the predictor. `likes` has a tiny coefficient (~10) but operates on a huge scale (millions); `publish_day_Sat` is just 0/1 so its coefficient looks bigger.
2. To compare predictor importance fairly, use **standardized coefficients**: $\beta \times \text{sd}(X) / \text{sd}(Y)$.
3. Alternatively, look at the t-statistic (or |t|) which automatically accounts for scale.

💡 *In multiple regression, raw coefficient magnitude alone is NOT a reliable measure of importance.*

In [ ]:
df_coefs = pd.DataFrame({
    'feature': X_tr.columns,
    'coef': model_d.coef_,
    'abs_coef': np.abs(model_d.coef_)
}).sort_values('abs_coef', ascending=False)
print(df_coefs)

# To assess importance properly:
print('\n--- Standardized coefficients (β × sd(X) / sd(y)) ---')
y_sd = y_tr.std()
for col, coef in zip(X_tr.columns, model_d.coef_):
    std_coef = coef * X_tr[col].std() / y_sd
    print(f'{col:<25}{std_coef:>10.4f}')

---
# 📋 Mock Quiz Answer Summary

| Q | Answer |
|---|---|
| 1 | `22832167.539130434` |
| 2 | Upward / Positive |
| 3 | likes, comments, number_of_tags, time_since |
| 4 | `length_description` |
| 5 | NOT significant (p ≈ 0.6) |
| 6 | Coefficient ≈ 135.40 (positive but not significant) |
| 7 | `22767003.534707773` |
| 8 | `22702824.52676366` |
| 9 | `1156276.181136` (5 × coefficient) |
| 10 | `0.12899783299327...` |
| 11 | `39525807.882388055` |
| 12 | Thu, Fri, Sat (Tue is borderline at p=0.0549!) |
| 13 | `279.5` |
| 14 | `0.23652173913043478` |
| 15 | `4999702.709397368` |
| 16 | `18123259.226605773` |
| 17 | `17554806.20928597` |
| 18 | `publish_day_Sat`; but largest coef ≠ most important |